In [12]:
from datasets import load_from_disk
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1021-2-2b_1p/xhl1021-2-2b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset')

OSError: Expected to be able to read 45338080 bytes for message body, got 45333664

In [ ]:
adata_emb = sc.read_h5ad('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_110m/07112025_054432_709/adata_ep3_5k_complete_processed.h5ad')

In [ ]:
adata_emb

In [18]:
adata_test = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhl1005-1b_1p/NSCLC_PT.h5ad')

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [1]:
import copy
import logging
import os
import pickle
import sys
import yaml
from collections import defaultdict
from pathlib import Path
from typing import Literal

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.multiprocessing as mp
import torch.nn.functional as F
from datasets import concatenate_datasets, Dataset
from functools import partial
from tqdm import tqdm
from pyensembl import EnsemblRelease

from app.helper import init_model, load_checkpoint
from nichejepa.datasets.cell_datasets import CellBaseDataset, init_cell_dataset
from nichejepa.datasets.dataloaders import init_dataloader_and_sampler
from nichejepa.masks.block_masking  import BlockMaskCollator
from nichejepa.masks.cell_masking import CellMaskCollator
from nichejepa.tokenizers import cell_tokenizers
from nichejepa.utils.embedding import (create_binary_selection_mask,
                                       compute_mean_unmasked_emb,
                                       compute_unmasked_rank_based_weights,
                                       collect_adata_from_folder,
                                       retrieve_gene_emb,
                                       compute_count_mean_cosine_sim,
                                       compute_sum_and_nonzero_count,
                                       batch_rowwise_distances)
from nichejepa.utils.logging import CSVLogger
from typing import Dict, List

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [57]:
ensembl_release = '109'

In [58]:
import pyensembl
pyensembl.__version__

'2.2.9'

In [59]:
ensembl = EnsemblRelease(release=ensembl_release, species='human')
ensembl.download()
ensembl.index()
all_genes = ensembl.genes()
protein_coding_genes = [
    gene for gene in all_genes if gene.biotype == "protein_coding"]
mirna_genes = [
    gene for gene in all_genes if gene.biotype == "miRNA"]
all_relevant_genes = protein_coding_genes + mirna_genes
gene_ensembl_map_dict = {
    gene.gene_name: gene.gene_id for gene in all_relevant_genes}

INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


In [60]:
len(gene_ensembl_map_dict)

21261

In [56]:
len(gene_ensembl_map_dict)

21249

In [ ]:
len(data)

In [3]:
import pickle
with open('/nfs/team361/sb75/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl', 'rb') as f:
    token_dict = pickle.load(f)

In [9]:
token_dict['spv_1020_batch3'] - 21957

801

In [14]:
token_dict['spv_1021_batch0'] - 21957

814

In [5]:
for key in token_dict.keys():
    if 'spv' in key:
        print(key)

spv_homo_sapiens
spv_brain
spv_breast
spv_lung
spv_pancreas
spv_skin
spv_heart
spv_lymph node
spv_bone
spv_colon
spv_kidney
spv_liver
spv_tonsil
spv_spinal_cord
spv_cervix
spv_prostate
spv_ovary
spv_uterus
spv_male_reproductive_system
spv_placenta
spv_xenium
spv_merfish
spv_cosmx
spv_cartana
spv_starmap
spv_gene_panel466
spv_gene_panel272
spv_gene_panel371
spv_gene_panel372
spv_gene_panel468
spv_gene_panel376
spv_gene_panel278
spv_gene_panel472
spv_gene_panel319
spv_gene_panel419
spv_gene_panel392
spv_gene_panel350
spv_gene_panel317
spv_gene_panel264
spv_gene_panel5028
spv_gene_panel4575
spv_gene_panel4954
spv_gene_panel470
spv_gene_panel412
spv_gene_panel361
spv_gene_panel4948
spv_gene_panel341
spv_gene_panel4949
spv_gene_panel312
spv_gene_panel380
spv_gene_panel498
spv_gene_panel394
spv_gene_panel6016
spv_gene_panel983
spv_gene_panel946
spv_gene_panel966
spv_gene_panel169
spv_gene_panel18893
spv_gene_panel237
spv_gene_panel413
spv_gene_panel297
spv_gene_panel289
spv_gene_panel949
spv

In [2]:
len(token_dict)

23407

In [4]:
token_dict['spt_batch']

106

In [7]:
token_dict['spt_gene_panel']

105

In [ ]:
with open("/lustre/scratch126/cellgen/lotfollahi/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl", 'rb') as f:
    gene_name_mapping_dict = pickle.load(f)

In [ ]:
token_dict_genes = [gene for gene in list(token_dict.keys()) if 'ENS' in gene]
mapping_dict_genes = list(gene_name_mapping_dict.values()) # list(data.values()) 

In [ ]:
len(gene_name_mapping_dict)

In [ ]:
gene_name_mapping_dict['CD99']

In [ ]:
token_dict['ENSG00000002586']

In [ ]:
set(mapping_dict_genes) - set(token_dict_genes)

In [ ]:
len(gene_name_mapping_dict)

In [ ]:
len(token_dict_genes)

In [ ]:
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl', 'rb') as f:
    token_dict2 = pickle.load(f)

In [ ]:
len(token_dict2)

In [ ]:
ensembl_ids_in_gene_dict = list(gene_ensembl_map_dict.values()) # list(data.values()) 

In [ ]:
ensembl_ids_in_token_dict = list(token_dict.keys())

In [ ]:
len(ensembl_ids_in_gene_dict)

In [ ]:
len(ensembl_ids_in_token_dict)

In [ ]:
missing_genes = [gene for gene in ensembl_ids_in_gene_dict if gene not in ensembl_ids_in_token_dict]
missing_genes_reverse = [gene for gene in ensembl_ids_in_token_dict if gene not in ensembl_ids_in_gene_dict]

In [ ]:
missing_genes

In [ ]:
[gene for gene in missing_genes_reverse if 'ENS' in gene]

In [ ]:
len(gene_ensembl_map_dict)

In [ ]:
import pickle

with open('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/models/01112025_081731_614/ensembl_dic.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
len(data)

In [ ]:
with open("/nfs/team361/sb75/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl", 'rb') as f:
    data2 = pickle.load(f)

In [ ]:
len(data2)

In [ ]:
len(data)

In [ ]:
import os
import scanpy as sc

In [ ]:
dataset_ids = ['2000']

In [ ]:
load_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver'

In [ ]:
for subdir, _, files in os.walk(load_folder_path):
    if dataset_ids:
        if not any(dataset_id in subdir.split('/')[-1].split('-')[0] for dataset_id in dataset_ids):
            continue
    print(f'Loading AnnData objects from {subdir}.')
    for file in files:
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

In [ ]:
adata = sc.read_h5ad('/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhp2000-1b_1p/Xenium_Pancreatic_Cancer_with_Xenium_Human_Multi-Tissue_and_Cancer_Panel.h5ad')

In [ ]:
adata

In [ ]:
sc.pl.spatial(adata, spot_size=20)

In [ ]:
from datasets import load_from_disk

In [ ]:
ds = load_from_disk('/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10.dataset')

In [ ]:
n_nz_tokens = ds['n_nonzero_tokens']

In [ ]:
import pickle

with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_full.pkl', "rb") as f: 
    n_nz_tokens_2 = pickle.load(f)


In [ ]:
n_nz_tokens[-10:]

In [ ]:
n_nz_tokens_2[-10:]

In [ ]:
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_6.dataset')

In [ ]:
datasets

In [ ]:
test_ds = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_ordered_full.dataset')

In [ ]:
test_ds

In [ ]:
del(test_ds)

In [ ]:
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_0.01_sample_32_None_None_None_None_None_shifted_log_knn_10_ordered.dataset')

In [ ]:
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10.dataset')

In [ ]:
dataset

In [ ]:
dataset_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
#train_idx_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl'
order_idx_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl'

#dataset_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
#train_idx_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl'
#order_idx_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx_split_train.pkl'

In [ ]:
dataset = load_from_disk(dataset_path)

In [ ]:
dataset

In [ ]:
import pickle

#with open(train_idx_path, "rb") as f: 
#    train_indices = pickle.load(f)

#dataset = dataset.select(train_indices)        

# Retrieve indices for reordering
with open(order_idx_path, "rb") as f: 
    order_indices = pickle.load(f)

dataset = dataset.select(order_indices)

In [ ]:
dataset

In [ ]:
print("Start flattening.")

# Materialize the new physical order
dataset = dataset.flatten_indices(
    num_proc=160,
    writer_batch_size=1000)

print("Flattening done.")

In [ ]:
dataset.data

In [ ]:
save_suffix = 'ordered_full'
new_path = dataset_path.replace('.dataset', f'_{save_suffix}.dataset')
print(new_path)
dataset.save_to_disk(new_path)

In [ ]:
2+3

In [ ]:
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_new.dataset')
dataset

In [ ]:
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_0.01_sample_32_None_None_None_None_None_shifted_log_knn_10_ordered_fixed.dataset')

In [ ]:
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset')

In [ ]:
dataset = load_from_disk('/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10.dataset')

In [ ]:
dataset

In [ ]:
dataset = dataset.remove_columns("batch_token")
dataset = dataset.remove_columns("gene_panel_token")
dataset = dataset.remove_columns("assay_token")
dataset = dataset.remove_columns("species_token")
dataset = dataset.remove_columns("tissue_token")

In [ ]:
dataset.save_to_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_new.dataset')

In [ ]:
d1 = load_from_disk('/nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_80m/18062025_082526_412/ep16_0_5k.dataset')

In [ ]:
d1

In [ ]:
dataset['n_nonzero_tokens'][-100:]

In [ ]:
rel_x_coord = dataset['rel_x_coord']

In [ ]:
len(rel_x_coord[0])

In [ ]:
dataset2 = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_fixed.dataset')

In [ ]:
train_mode = True

def set_minimal_format(d):
    cols = [c for c in d.column_names if c != 'cell_id' or not train_mode]
    d.set_format(type="torch", columns=cols, output_all_columns=False)
    return d

dataset2 = set_minimal_format(dataset2)

In [ ]:
dataset2

In [ ]:
item = dataset2[0]

In [ ]:
item

In [ ]:
'seg_tokens' in item.keys()

In [ ]:
nz_tokens = list(dataset['n_nonzero_tokens'])

In [ ]:
nz_tokens[-100:]

In [ ]:
test_batch_ids = ['49_batch0',
                                 '59_batch0',
                                 '59_batch3',
                                 '59_batch6',
                                 '59_batch7',
                                 '59_batch2',
                                 '59_batch4',
                                 '59_batch1',
                                 '59_batch5',
                                 '78_batch3',
                                 '78_batch10',
                                 '78_batch1',
                                 '78_batch0',
                                 '78_batch2',
                                 '78_batch6',
                                 '78_batch7',
                                 '78_batch9',
                                 '78_batch8',
                                 '78_batch4',
                                 '78_batch5',
                                 '1000_batch4',
                                 '1000_batch6',
                                 '1000_batch11',
                                 '1000_batch12',
                                 '1000_batch18',
                                 '1000_batch19',
                                 '1000_batch21',
                                 '1000_batch22',
                                 '1000_batch34',
                                 '1000_batch38',
                                 '1000_batch20',
                                 '1000_batch26',
                                 '1000_batch27',
                                 '1000_batch32',
                                 '1000_batch36',
                                 '1002_batch3',
                                 '1002_batch15'
                                 '1002_batch27',
                                 '1002_batch39',
                                 '1002_batch45',
                                 '1002_batch26',
                                 '1002_batch4',
                                 '1002_batch16'
                                 '1002_batch18',
                                 '1002_batch23',
                                 '1002_batch31',
                                 '1002_batch34',
                                 '1002_batch53',
                                 '1002_batch55',
                                 '1002_batch40',
                                 '1002_batch46',
                                 '1007_batch1',
                                 '1007_batch14',
                                 '1007_batch20',
                                 '1007_batch36',
                                 '1007_batch17',
                                 '1007_batch33',
                                 '1007_batch4',
                                 '1007_batch2',
                                 '1007_batch0',
                                 '1007_batch27',
                                 '1007_batch3',
                                 '1007_batch12',
                                 '1007_batch19',
                                 '1007_batch24',
                                 '1007_batch6',
                                 '1007_batch25',
                                 '1007_batch34',
                                 '1007_batch30',
                                 '1007_batch39',
                                 '1007_batch22',
                                 '1009_batch0',
                                 '1009_batch1',
                                 '1009_batch2',
                                 '1009_batch3',
                                 '1009_batch4',
                                 '1009_batch5',
                                 '1009_batch6',
                                 '1009_batch7',
                                 '1009_batch8',
                                 '1009_batch9',
                                 '1009_batch10',
                                 '1009_batch11',
                                 '1009_batch12',
                                 '1009_batch13',
                                 '1009_batch14',
                                 '1009_batch15',
                                 '1009_batch16',
                                 '1011_batch0',
                                 '1011_batch1',
                                 '1011_batch2',
                                 '1011_batch3',
                                 '1011_batch4',
                                 '1011_batch5',
                                 '1011_batch6',
                                 '1011_batch7',
                                 '1011_batch8',
                                 '1011_batch9',
                                 '1011_batch10',
                                 '1011_batch11',
                                 '1011_batch12',
                                 '1011_batch13',
                                 '1011_batch14',
                                 '1011_batch15',
                                 '1011_batch16',
                                 '1011_batch17',
                                 '1011_batch18',
                                 '1011_batch19',
                                 '1011_batch20',
                                 '1011_batch21']

In [ ]:
test_batch_ids

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import pickle

import numpy as np
import torch
from datasets import load_from_disk

from nichejepa.datasets.cell_neighborhood_dataset import make_cell_neighborhood_dataset 
from nichejepa.helper import init_model
from nichejepa.masks.multigene import MaskCollator
from nichejepa.utils.distributed import init_distributed

from logging import getLogger
from multiprocessing import Value

import torch.nn.functional as F

from nichejepa.masks.utils import apply_masks
from nichejepa.utils.tensors import repeat_interleave_batch

In [ ]:
def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    """
    embed_dim: output dimension for each position
    pos: a list of positions to be encoded: size (M,)
    out: (M, D)
    """
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=float)
    omega /= embed_dim / 2.
    omega = 1. / 10000**omega   # (D/2,)

    pos = pos.reshape(-1)   # (M,)
    out = np.einsum('m,d->md', pos, omega)   # (M, D/2), outer product

    emb_sin = np.sin(out)  # (M, D/2)
    emb_cos = np.cos(out)  # (M, D/2)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)
    return emb

In [ ]:
def get_1d_sincos_pos_embed(embed_dim, grid_size, cls_token=False):
    """
    grid_size: int of the grid length
    return:
    pos_embed: [grid_size, embed_dim] or [1+grid_size, embed_dim] (w/ or w/o cls_token)
    """
    grid = np.arange(grid_size, dtype=float)
    pos_embed = get_1d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token:
        pos_embed = np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
    return pos_embed

In [ ]:
pos_embed = get_1d_sincos_pos_embed(embed_dim=4, grid_size=10, cls_token=False)

In [ ]:
pos_embed

In [ ]:
def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0

    # use half of dimensions to encode grid_h
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)

    emb = np.concatenate([emb_h, emb_w], axis=1)  # (H*W, D)
    return emb

In [ ]:
def get_2d_sincos_pos_embed(embed_dim, grid_size, cls_token=False):
    """
    grid_size: int of the grid height and width
    return:
    pos_embed: [grid_size*grid_size, embed_dim] or [1+grid_size*grid_size, embed_dim] (w/ or w/o cls_token)
    """
    grid_h = np.arange(grid_size, dtype=float)
    grid_w = np.arange(grid_size, dtype=float)
    grid = np.meshgrid(grid_w, grid_h)  # here w goes first
    grid = np.stack(grid, axis=0)

    grid = grid.reshape([2, 1, grid_size, grid_size])
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token:
        pos_embed = np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
    return pos_embed

In [ ]:
pos_embed = get_2d_sincos_pos_embed(embed_dim=4, grid_size=10, cls_token=False)

In [ ]:
pos_embed.shape

In [ ]:
grid_size = 10

grid_h = np.arange(grid_size, dtype=float)
grid_w = np.arange(grid_size, dtype=float)
grid = np.meshgrid(grid_w, grid_h)  # here w goes first

In [ ]:
grid = np.stack(grid, axis=0)

In [ ]:
grid

In [ ]:
grid = grid.reshape([2, 1, grid_size, grid_size])

In [ ]:
grid

In [ ]:
x = torch.tensor([[2, 3, 5], [6,7,8]])

In [ ]:
shape = (x.shape[0],) + (1,) * (x.ndim - 1) 

In [ ]:
shape

In [ ]:
x.shape[0]

In [ ]:
(x.shape[0],) + (1,)

In [ ]:
(1,) * (x.ndim - 1) 

In [ ]:
None or 200

In [ ]:
drop_path_rate = 0.5
depth = 10

dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]

In [ ]:
dpr

In [ ]:
batch = (torch.tensor([1, 2]), torch.tensor([3, 4]))

In [ ]:
batch

In [ ]:
torch.utils.data.default_collate(batch)

In [ ]:
from multiprocessing import Value

In [ ]:
Value('i', -1)

In [ ]:
g = torch.Generator()

In [ ]:
def _sample_block_size(self, generator, scale, aspect_ratio_scale):
    _rand = torch.rand(1, generator=generator).item()
    # -- Sample block scale
    min_s, max_s = scale
    mask_scale = min_s + _rand * (max_s - min_s)
    max_keep = int(self.height * self.width * mask_scale)
    # -- Sample block aspect-ratio
    min_ar, max_ar = aspect_ratio_scale
    aspect_ratio = min_ar + _rand * (max_ar - min_ar)
    # -- Compute block height and width (given scale and aspect-ratio)
    h = int(round(math.sqrt(max_keep * aspect_ratio)))
    w = int(round(math.sqrt(max_keep / aspect_ratio)))
    while h >= self.height:
        h -= 1
    while w >= self.width:
        w -= 1

    return (h, w)

In [ ]:
_rand = torch.rand(1).item()

In [ ]:
_rand

In [ ]:
def apply_masks(x, masks):
    """
    :param x: tensor of shape [B (batch-size), N (num-patches), D (feature-dim)]
    :param masks: list of tensors containing indices of patches in [N] to keep
    """
    all_x = []
    for m in masks:
        mask_keep = m.unsqueeze(-1).repeat(1, 1, x.size(-1))
        all_x += [torch.gather(x, dim=1, index=mask_keep)]
    return torch.cat(all_x, dim=0)

In [ ]:
x = torch.tensor([
    [[1, 2, 1, 2], [3, 4, 3, 1], [5, 6, 7, 2]],  # Batch 1
    [[7, 8, 2, 1], [9, 10, 5, 3], [11, 12, 8, 4]]  # Batch 2
])
masks = [torch.tensor([0, 2])] # , torch.tensor([1])]

In [ ]:
x

In [ ]:
for m in masks:
    mask_keep = m.unsqueeze(-1).repeat(1, 1, x.size(-1))

In [ ]:
mask_keep

In [ ]:
all_x = []
all_x += [torch.gather(x, dim=1, index=mask_keep)]

In [ ]:
x

In [ ]:
mask_keep

In [ ]:
all_x

In [ ]:
torch.cat(all_x, dim=0)

In [ ]:
batch

In [ ]:
B = len(batch)

In [ ]:
collated_batch = torch.utils.data.default_collate(batch)

In [ ]:
collated_batch

In [ ]:
import math

def _sample_block_size(generator, scale, aspect_ratio_scale):

    height = 224 / 16
    width = 224 / 16

    print(height)

    _rand = torch.rand(1, generator=generator).item()
    print(_rand)
    # -- Sample block scale
    min_s, max_s = scale
    mask_scale = min_s + _rand * (max_s - min_s)
    print(mask_scale)
    max_keep = int(height * width * mask_scale)
    print(max_keep)
    # -- Sample block aspect-ratio
    min_ar, max_ar = aspect_ratio_scale
    aspect_ratio = min_ar + _rand * (max_ar - min_ar)
    # -- Compute block height and width (given scale and aspect-ratio)
    h = int(round(math.sqrt(max_keep * aspect_ratio)))
    w = int(round(math.sqrt(max_keep / aspect_ratio)))
    print(h)
    print(w)
    print(aspect_ratio)
    while h >= height:
        h -= 1
    while w >= width:
        w -= 1

    return (h, w)

In [ ]:
g = torch.Generator()
pred_mask_scale=(0.2, 0.8)
aspect_ratio=(0.3, 3.0)

p_size = _sample_block_size(
            generator=g,
            scale=pred_mask_scale,
            aspect_ratio_scale=aspect_ratio)

In [ ]:
top = torch.randint(0, 12, (1,))

In [ ]:
def _sample_block_mask(b_size, acceptable_regions=None):
    h, w = b_size

    height = int(224 / 16)
    width = int(224 / 16)
    min_keep = 4

    def constrain_mask(mask, tries=0):
        """ Helper to restrict given mask to a set of acceptable regions """
        N = max(int(len(acceptable_regions)-tries), 0)
        for k in range(N):
            mask *= acceptable_regions[k]
    # --
    # -- Loop to sample masks until we find a valid one
    tries = 0
    timeout = og_timeout = 20
    valid_mask = False
    while not valid_mask:
        # -- Sample block top-left corner
        top = torch.randint(0, height - h, (1,))
        left = torch.randint(0, width - w, (1,))
        mask = torch.zeros((height, width), dtype=torch.int32)
        mask[top:top+h, left:left+w] = 1
        # -- Constrain mask to a set of acceptable regions
        if acceptable_regions is not None:
            constrain_mask(mask, tries)
        mask = torch.nonzero(mask.flatten())
        # -- If mask too small try again
        valid_mask = len(mask) > min_keep
        if not valid_mask:
            timeout -= 1
            if timeout == 0:
                tries += 1
                timeout = og_timeout
                logger.warning(f'Mask generator says: "Valid mask not found, decreasing acceptable-regions [{tries}]"')
    mask = mask.squeeze()
    # --
    mask_complement = torch.ones((height, width), dtype=torch.int32)
    mask_complement[top:top+h, left:left+w] = 0
    # --
    return mask, mask_complement

In [ ]:
mask, mask_c = _sample_block_mask(p_size, acceptable_regions=None)

In [ ]:
mask

In [ ]:
mask_c

In [ ]:
batch

In [ ]:
g = torch.Generator()
pred_mask_scale=(0.15, 0.2)
aspect_ratio=(0.75, 1.5)
enc_mask_scale=(0.85, 1.0)
height = int(224/16)
width = int(224/16)
npred = 4
nenc = 1
allow_overlap = False

def call_mask(batch):
    '''
    Create encoder and predictor masks when collating imgs into a batch
    # 1. sample enc block (size + location) using seed
    # 2. sample pred block (size) using seed
    # 3. sample several enc block locations for each image (w/o seed)
    # 4. sample several pred block locations for each image (w/o seed)
    # 5. return enc mask and pred mask
    '''
    B = len(batch)
    
    collated_batch = torch.utils.data.default_collate(batch)
    
    seed = 42
    g = torch.Generator()
    g.manual_seed(seed)
    p_size = _sample_block_size(
        generator=g,
        scale=pred_mask_scale,
        aspect_ratio_scale=aspect_ratio)
    e_size = _sample_block_size(
        generator=g,
        scale=enc_mask_scale,
        aspect_ratio_scale=(1., 1.))
    
    collated_masks_pred, collated_masks_enc = [], []
    min_keep_pred = height * width
    min_keep_enc = height * width
    for _ in range(B):
    
        masks_p, masks_C = [], []
        for _ in range(npred):
            mask, mask_C = _sample_block_mask(p_size)
            masks_p.append(mask)
            masks_C.append(mask_C)
            min_keep_pred = min(min_keep_pred, len(mask))
        collated_masks_pred.append(masks_p)
    
        acceptable_regions = masks_C
        try:
            if allow_overlap:
                acceptable_regions= None
        except Exception as e:
            logger.warning(f'Encountered exception in mask-generator {e}')
    
        masks_e = []
        for _ in range(nenc):
            mask, _ = _sample_block_mask(e_size, acceptable_regions=acceptable_regions)
            masks_e.append(mask)
            min_keep_enc = min(min_keep_enc, len(mask))
        collated_masks_enc.append(masks_e)

    print(collated_masks_enc[0][0].shape)
    print(collated_masks_enc[1][0].shape)
    
    collated_masks_pred = [[cm[:min_keep_pred] for cm in cm_list] for cm_list in collated_masks_pred]
    collated_masks_pred = torch.utils.data.default_collate(collated_masks_pred)
    # --
    collated_masks_enc = [[cm[:min_keep_enc] for cm in cm_list] for cm_list in collated_masks_enc]
    collated_masks_enc = torch.utils.data.default_collate(collated_masks_enc)

    print(collated_masks_enc[0].shape)
    
    return collated_batch, collated_masks_enc, collated_masks_pred

In [ ]:
batch = [torch.randint(255, (224, 224)), torch.randint(255, (224, 224))]

collated_batch, collated_masks_enc, collated_masks_pred = call_mask(batch)

In [ ]:
collated_batch

In [ ]:
collated_masks_enc

In [ ]:
collated_masks_pred

In [ ]:
def apply_masks(x, masks):
    """
    :param x: tensor of shape [B (batch-size), N (num-patches), D (feature-dim)]
    :param masks: list of tensors containing indices of patches in [N] to keep
    """
    all_x = []
    for m in masks:
        mask_keep = m.unsqueeze(-1).repeat(1, 1, x.size(-1))
        all_x += [torch.gather(x, dim=1, index=mask_keep)]
    return torch.cat(all_x, dim=0)

In [ ]:
x = torch.randn(2, 14*14, 4)

In [ ]:
test = apply_masks(x, collated_masks_enc)

In [ ]:
test.shape

In [ ]:
import torch

# Input tensor x of shape [2, 5, 3]
x = torch.tensor([
    [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12], [13, 14, 15]],
    [[16, 17, 18], [19, 20, 21], [22, 23, 24], [25, 26, 27], [28, 29, 30]]
])

# Masks
masks = [torch.tensor([0, 2]), torch.tensor([1, 3])]

# Applying the function
result = apply_masks(x, masks)
print(result)

In [ ]:
torch.zeros(3, dtype=torch.int32)

In [ ]:
tokenizer = "cell_neighborhood_tokenizer"
st_data_folder_path = "../datasets/st_data" # spatially resolved transcriptomics data
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
norm_factor = "read_depth"
norm_method = "shifted_log"
dataset_name = "nanostring_cosmx_human_brain"
model_input_size = 2048

# Load tokenized dataset
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

dataset = dataset.train_test_split(test_size=0.10, seed=42)

In [ ]:
tokenizer_folder_path = f"../datasets/tokenizer/{tokenizer}"
nichejepa_token_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/token_dictionary.pkl"

# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

In [ ]:
tokens = [token for token in token_dict.keys() if "neighborhood" not in token]

In [ ]:
len(set(tokens))

In [ ]:
cndataset, data_loader, dist_sampler = make_cell_neighborhood_dataset(
    16, dataset["train"])
print(cndataset[0])
batch = [cndataset[0], cndataset[1]]
batch[0][9] = 0
print(batch)

In [ ]:
B = len(batch)
collated_batch = torch.utils.data.default_collate(batch)
g = torch.Generator()

In [ ]:
_GLOBAL_SEED = 0
logger = getLogger()


class MaskCollator(object):
    def __init__(self,
                 seq_len=10,
                 n_targets=2,
                 n_contexts=1):
        super(MaskCollator, self).__init__()
        self.seq_len = seq_len
        self.n_targets = n_targets
        self.n_contexts = n_contexts
        self._itr_counter = Value('i', -1)  # collator is shared across worker processes

    def step(self):
        i = self._itr_counter
        with i.get_lock():
            i.value += 1
            v = i.value
        return v
    
    def _sample_gene_mask(self,
                          non_zero_seq_len,
                          mask_size,
                          valid_token_masks=None):
        if mask_size < non_zero_seq_len:
            # -- Sample start token
            start = torch.randint(0,
                                  non_zero_seq_len - mask_size,
                                  (1,))
        else:
            start = 0

        mask = torch.zeros(self.seq_len, dtype=torch.int32)
        mask[start:start+mask_size] = 1

        # -- Constrain mask to a set of valid tokens
        if valid_token_masks is not None:
            for k in range(len(valid_token_masks)):
                mask *= valid_token_masks[k]

        mask = torch.nonzero(mask.flatten())
        mask = mask.squeeze()
        # --
        mask_complement = torch.ones(self.seq_len, dtype=torch.int32)
        mask_complement[start:start+mask_size] = 0
        # --
        return mask, mask_complement

    def __call__(self, batch):
        '''
        Create context and target masks when collating cell neighborhoods into a batch
        # 1. sample context block (size + location) using seed
        # 2. sample target block (size) using seed
        # 3. sample several context block locations for each cell neighborhood (w/o seed)
        # 4. sample several target block locations for each cell neighborhood (w/o seed)
        # 5. return context mask and target mask
        '''
        B = len(batch)

        collated_batch = torch.utils.data.default_collate(batch)

        seed = self.step()
        g = torch.Generator()
        g.manual_seed(seed)

        collated_masks_context, collated_masks_target = [], []
        target_mask_size = 2 # TODO change
        context_mask_size = 10 # TODO change

        keep_tokens_target = self.seq_len
        keep_tokens_context = self.seq_len
        for i in range(B):
            masks_target_complement = []
            masks_target = []
            masks_context = []

            non_zero_seq_len = torch.nonzero(batch[i]).size(0)
            print(non_zero_seq_len)

            for _ in range(self.n_targets):
                mask_target, mask_target_complement = self._sample_gene_mask(
                    non_zero_seq_len,
                    target_mask_size)
                masks_target.append(mask_target)
                masks_target_complement.append(mask_target_complement)
                keep_tokens_target = min(keep_tokens_target, len(mask_target))
            
            for _ in range(self.n_contexts):
                mask_context, _ = self._sample_gene_mask(
                    non_zero_seq_len,
                    context_mask_size,
                    valid_token_masks=masks_target_complement)
                masks_context.append(mask_context)
                keep_tokens_context = min(keep_tokens_context, len(mask_context))

            collated_masks_target.append(masks_target)
            collated_masks_context.append(masks_context)

        collated_masks_target = [[cm[:keep_tokens_target] for cm in cm_list] for cm_list in collated_masks_target]
        collated_masks_target = torch.utils.data.default_collate(collated_masks_target)
        collated_masks_context = [[cm[:keep_tokens_context] for cm in cm_list] for cm_list in collated_masks_context]
        collated_masks_context = torch.utils.data.default_collate(collated_masks_context)

        return collated_batch, collated_masks_context, collated_masks_target


In [ ]:
mask_col = MaskCollator()
collated_batch, collated_masks_context, collated_masks_target = mask_col(batch)
print(collated_masks_context)
print(collated_masks_target)

In [ ]:
# config
batch_size = 4
vocab_size = 6029
seq_len = 10
n_targets = 2
n_contexts = 1
pin_mem = True
num_workers = 1
pred_depth = 2
pred_emb_dim = 6
model_name = "gt_tiny"

In [ ]:
world_size, rank = init_distributed()

In [ ]:
mask_collator = MaskCollator(seq_len=seq_len,
                             n_targets=n_targets,
                             n_contexts=n_contexts)

In [ ]:
mask_collator.target_mask_size

In [ ]:
_, unsupervised_loader, unsupervised_sampler = make_cell_neighborhood_dataset(
        batch_size=batch_size,
        data=dataset["train"],
        vocab_size=vocab_size, 
        seq_len=seq_len,
        collator=mask_collator,
        pin_mem=pin_mem,
        training=True,
        num_workers=num_workers,
        world_size=world_size,
        rank=rank,
        drop_last=True)
ipe = len(unsupervised_loader)

In [ ]:
#device = torch.device('cuda:0')
device = torch.device('cpu')
#torch.cuda.set_device(device)

In [ ]:
for itr, (udata, masks_enc, masks_pred) in enumerate(unsupervised_loader):
    def load_cell_neighborhoods():
        # -- unsupervised imgs
        cell_neighborhood_tokens = udata.to(device, non_blocking=True)
        masks_1 = [u.to(device, non_blocking=True) for u in masks_enc]
        masks_2 = [u.to(device, non_blocking=True) for u in masks_pred]
        return (cell_neighborhood_tokens, masks_1, masks_2)
    cell_neighborhood_tokens, masks_enc, masks_pred = load_cell_neighborhoods()
    break

In [ ]:
cell_neighborhood_tokens

In [ ]:
encoder, predictor = init_model(
    device=device,
    seq_len=seq_len,
    pred_depth=pred_depth,
    pred_emb_dim=pred_emb_dim,
    model_name=model_name)
target_encoder = copy.deepcopy(encoder)

In [ ]:
next(target_encoder.parameters()).device

In [ ]:
next(encoder.parameters()).device

In [ ]:
cell_neighborhood_tokens.shape # B=4, seq_len=10

In [ ]:
cell_neighborhood_tokens

In [ ]:
h = target_encoder(cell_neighborhood_tokens)

In [ ]:
h.shape # B=4, seq_len=10, emb_siz=12

In [ ]:
h = F.layer_norm(h, (h.size(-1),))

In [ ]:
h.shape # B=4, seq_len=10, emb_siz=12

In [ ]:
B = len(h)

In [ ]:
h = apply_masks(h, masks_pred)

In [ ]:
masks_pred[0].shape # B=4, target_size=3

In [ ]:
masks_pred

In [ ]:
masks_pred # n_targets=2 (list), B=4, target_size=3

In [ ]:
h.shape # B*n_targets=8, target_size=3, emb_size=12

In [ ]:
h = repeat_interleave_batch(h, B, repeat=len(masks_enc))

In [ ]:
h.shape

In [ ]:
cell_neighborhood_tokens.shape

In [ ]:
cell_neighborhood_tokens

In [ ]:
h

In [ ]:
z = encoder(cell_neighborhood_tokens, masks_enc)

In [ ]:
z.shape

In [ ]:
z = predictor(z, masks_enc, masks_pred)

In [ ]:
z.shape

In [ ]:
loss = F.smooth_l1_loss(z, h)

In [ ]:
loss

In [ ]:
test = {'params': (p for n, p in encoder.named_parameters()
                    if ('bias' not in n) and (len(p.shape) != 1))}

In [ ]:
next(test["params"])

In [ ]:
import pickle

datasets_folder_path = '/nfs/team361/sb75/DATASETS' # '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = f'{datasets_folder_path}/tokenizer/token_dictionary_homo_sapiens.pkl'
token_dict_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl'

# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
max_cls_tokens = sum(1 for key in token_dict if "cls" in key)
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

In [ ]:
token_dict['spv_1020_batch14']

In [ ]:
token_dict_path

In [ ]:
idx_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_0.1_sample_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl'

with open(idx_path, 'rb') as f:
    idx = pickle.load(f)

In [ ]:
max(idx)

In [ ]:
import argparse
import pickle
import time
import timeit
import random

import torch
from datasets import load_from_disk

from nichejepa.masks.block_masking  import BlockMaskCollator
from nichejepa.datasets.cell_datasets import init_cell_dataset
from nichejepa.datasets.dataloaders import init_dataloader_and_sampler, CustomDistributedLengthGroupedSampler

In [ ]:
dataset_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_0.25_sample_32_None_None_None_None_None_shifted_log_knn_10.dataset'
nz_tokens_path = 'None'
idx_save_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_0.25_sample_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl'
seq_len_cell = 256
seq_len_neighborhood = 2560
gt_type = 'combined'
cell_pos_enc = 'coord'
n_targets = 1
n_contexts = 1
n_segments = 11
n_special_tokens = 0
per_block_mask_ratio = 0.6
batch_size = 124

In [ ]:
# Load Huggingface dataset
dataset = load_from_disk(dataset_path)
print(dataset)

# Load precomputed nonzero tokens
if nz_tokens_path == "None":
    nz_tokens_path = None

if nz_tokens_path:
    with open(nz_tokens_path, "rb") as f: 
        n_nonzero_tokens = pickle.load(f)
else:
    n_nonzero_tokens = None

# Set params
token_dict_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl'

with open(token_dict_folder_path, 'rb') as file:
    token_dict = pickle.load(file)
vocab_size = len(token_dict)

special_tokens = []
sampling_strategy = None
sep_gene_tokens_neb = False

In [ ]:
### Initialize dataloader and sampler ###
# Initialize cell dataset
cell_dataset = init_cell_dataset(
    dataset=dataset,
    vocab_size=vocab_size,
    seq_len_cell=seq_len_cell,
    seq_len_neighborhood=seq_len_neighborhood,
    tokenizer_type='cell_graph',
    gt_type=gt_type,
    cell_pos_enc=cell_pos_enc,
    special_tokens=special_tokens,
    sampling_strategy=sampling_strategy,
    n_nonzero_tokens_list=n_nonzero_tokens,
    include_cell_id=False,
    sep_gene_tokens_neb=sep_gene_tokens_neb)

# Initialize mask collator
mask_collator = BlockMaskCollator(
    n_targets=n_targets,
    n_contexts=n_contexts,
    n_segments=n_segments,
    seq_len_cell=seq_len_cell,
    seq_len_neighborhood=seq_len_neighborhood,
    n_special_tokens=n_special_tokens,
    per_block_mask_ratio=per_block_mask_ratio,
    sample_segments=True,
    sample_gene_masks=True)

# Initialize loader and sampler
distributed = True # True, False
world_size = 1
rank = 0
pin_memory = True
num_workers = 6

if not distributed:
    loader, sampler = init_dataloader_and_sampler(
        cell_dataset=cell_dataset,
        batch_size=batch_size,
        distributed=distributed,
        world_size=world_size,
        rank=rank,
        collate_fn=mask_collator,
        pin_memory=pin_memory,
        num_workers=num_workers,
        drop_last=True,
        #prefetch_factor=4,
        persistent_workers=False)
else:
    loader, sampler = init_dataloader_and_sampler(
        cell_dataset=cell_dataset,
        batch_size=batch_size,
        distributed=distributed,
        world_size=world_size,
        rank=rank,
        collate_fn=mask_collator,
        pin_memory=pin_memory,
        num_workers=num_workers,
        drop_last=True,
        #prefetch_factor=4,
        persistent_workers=False)
###############################

In [ ]:
### Sample length grouped indices ###
g = torch.Generator()
g.manual_seed(0)
indices = sampler._get_length_grouped_indices(generator=g)
#####################################

In [ ]:
with open(idx_save_path, "wb") as f: 
    pickle.dump(indices, f)

In [ ]:
import scanpy as sc
adata_gps = sc.read_h5ad('/nfs/team361/dj17/RCC_nicheJEPA/GPannotation/myeloid_v1/macrophageGPs_v1_annotated.h5ad')

In [ ]:
adata_gps.obs

In [ ]:
sc.pp.neighbors(adata_gps, use_rep='X')
sc.tl.umap(adata_gps)

In [ ]:
sc.pl.umap(adata_gps,
           color='gene_clusters_annotated',
           show=False)

In [ ]:
dataset = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered.dataset')

In [ ]:
dataset

In [ ]:
n_nonzero_tokens = list(dataset['n_nonzero_tokens'])

In [ ]:
n_nonzero_tokens[:1000]

In [ ]:
tokenizer_folder_path = '/nfs/team361/sb75/DATASETS/tokenizer'

In [ ]:
tokenized_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'

In [ ]:
f'{tokenizer_folder_path}/{tokenized_data_folder_path.rsplit("/", 1)[-1].removesuffix(".dataset")}_precomputed_split_train.pkl'

In [ ]:
import pickle
with open('/nfs/team361/sb75/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', "rb") as f: 
    train_idx = pickle.load(f)

In [ ]:
import os
import scanpy as sc

load_folder_path = '/nfs/team361/sb75/DATASETS/silver/xhb1002-AT15-10b_1p'

for subdir, _, files in os.walk(load_folder_path):
    print(f'Loading AnnData objects from {subdir}.')
    for file in files:
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(adata.uns['batch'])